# Rename Training WAV Files

This notebook recursively previews and renames training `.wav` files to:

`[xx]-[type]-[yy]-[rec]-[zzzz].wav`

- `xx`: dB level (`-6`, `+0`, `+6`)
- `type`: machine type (e.g. `pump`, `valve`, `fan`, `slider`)
- `yy`: numeric part of `id_nn`
- `rec`: `ab` or `nm` from `abnormal/normal`
- `zzzz`: last 4 chars of original filename stem

The parser supports folder variants such as `+0db-fan/id_00/...` and `+6_dB_fan/fan/id_00/...`.
Renaming executes only when `PROCESS = True`.


In [7]:
from pathlib import Path
import re
from uuid import uuid4

import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, display
from ipyfilechooser import FileChooser

PROCESS = True

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'preprocessing').exists():
            return candidate
    raise RuntimeError('Could not infer repo root. Please open notebook from inside the repo tree.')


def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, Path.cwd().resolve()]:
        if candidate.exists():
            return candidate
    return Path.cwd().resolve()


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEFAULT_DATA_ROOT = REPO_ROOT / 'datasets'
DISPLAY_LIMIT = 200

input_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_DATA_ROOT)))
input_dir_chooser.title = '<b>Input directory (db/type -> id_nn -> normal|abnormal)</b>'
input_dir_chooser.show_only_dirs = True
input_dir_chooser.use_dir_icons = True
input_dir_chooser.select_default = True

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))
display(input_dir_chooser)
print(f'PROCESS: {PROCESS}')
print(f'DISPLAY_LIMIT: {DISPLAY_LIMIT}')


Using repo root: `/home/mitch/development/raccoon-ball`

FileChooser(path='/home/mitch/development/raccoon-ball/datasets', filename='', title='<b>Input directory (db/t…

PROCESS: True
DISPLAY_LIMIT: 200


In [8]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()


DB_TYPE_TOKEN_RE = re.compile(r'([+-]?\d+)[_-]?d[bB][_-]?(.*)')
ID_DIR_RE = re.compile(r'id_(\d+)', re.IGNORECASE)
RENAMED_FILE_RE = re.compile(
    r'([+-]?\d+)-([A-Za-z0-9_-]+)-(\d+)-(ab|nm)-([A-Za-z0-9]{4})\.wav$',
    re.IGNORECASE,
)


def _normalise_machine_type(machine_type: str) -> str:
    normalised = re.sub(r'[^A-Za-z0-9]+', '-', machine_type.strip().lower())
    return normalised.strip('-')


def _parse_db_and_type(token: str) -> tuple[str | None, str | None]:
    m = DB_TYPE_TOKEN_RE.fullmatch(token.strip())
    if not m:
        return None, None

    db_raw, machine_type_raw = m.groups()
    xx = f'{int(db_raw):+d}'
    machine_type = _normalise_machine_type(machine_type_raw)
    if machine_type == '':
        machine_type = None
    return xx, machine_type


def _parse_id(id_dir: str) -> str:
    m = ID_DIR_RE.fullmatch(id_dir)
    if not m:
        raise ValueError(f'Expected id_<digits>, got: {id_dir}')
    return m.group(1)


def _parse_rec(rec_dir: str) -> str:
    rec_map = {'normal': 'nm', 'abnormal': 'ab'}
    key = rec_dir.lower()
    if key not in rec_map:
        raise ValueError(f'Expected normal/abnormal directory, got: {rec_dir}')
    return rec_map[key]


def _find_id_and_rec_indices(parts: tuple[str, ...]) -> tuple[int, str, str]:
    found_idx = None
    for i in range(len(parts) - 1):
        if ID_DIR_RE.fullmatch(parts[i]) and parts[i + 1].lower() in {'normal', 'abnormal'}:
            found_idx = i

    if found_idx is None:
        raise ValueError('Could not locate id_<nn>/(normal|abnormal) segment in path')

    return found_idx, parts[found_idx], parts[found_idx + 1]


def _extract_components_from_full_path(wav_path: Path) -> tuple[str, str, str, str]:
    parts = wav_path.resolve().parts
    id_idx, id_dir, rec_dir = _find_id_and_rec_indices(parts)

    yy = _parse_id(id_dir)
    rec = _parse_rec(rec_dir)

    db_idx = None
    xx = None
    machine_type = None

    for i in range(id_idx - 1, -1, -1):
        parsed_xx, parsed_type = _parse_db_and_type(parts[i])
        if parsed_xx is not None:
            db_idx = i
            xx = parsed_xx
            machine_type = parsed_type
            break

    if xx is None:
        raise ValueError(f'Could not locate a <db> token before {id_dir} in path: {wav_path}')

    if not machine_type:
        for i in range(id_idx - 1, db_idx, -1):
            candidate = parts[i]
            if ID_DIR_RE.fullmatch(candidate):
                continue
            if candidate.lower() in {'normal', 'abnormal'}:
                continue
            parsed_xx, _ = _parse_db_and_type(candidate)
            if parsed_xx is not None:
                continue

            machine_type = _normalise_machine_type(candidate)
            if machine_type:
                break

    if not machine_type:
        raise ValueError(f'Could not infer machine type from path: {wav_path}')

    return xx, machine_type, yy, rec


def _parse_components_from_renamed_filename(filename: str) -> dict | None:
    m = RENAMED_FILE_RE.fullmatch(filename)
    if not m:
        return None

    xx_raw, machine_type_raw, yy, rec, zzzz = m.groups()
    return {
        'xx': f'{int(xx_raw):+d}',
        'type': _normalise_machine_type(machine_type_raw),
        'yy': yy,
        'rec': rec.lower(),
        'zzzz': zzzz,
    }


def _build_new_name(wav_path: Path) -> tuple[str, dict]:
    stem = wav_path.stem
    zzzz = stem[-4:] if len(stem) >= 4 else stem.zfill(4)

    try:
        xx, machine_type, yy, rec = _extract_components_from_full_path(wav_path)
        new_name = f'{xx}-{machine_type}-{yy}-{rec}-{zzzz}.wav'
        parts = {'xx': xx, 'type': machine_type, 'yy': yy, 'rec': rec, 'zzzz': zzzz}
        return new_name, parts
    except Exception:
        existing_parts = _parse_components_from_renamed_filename(wav_path.name)
        if existing_parts is not None:
            return wav_path.name, existing_parts
        raise


DATA_ROOT = resolve_selected_dir(input_dir_chooser, DEFAULT_DATA_ROOT)
print(f'DATA_ROOT: {DATA_ROOT}')

data_root_resolved = DATA_ROOT.resolve()
wav_files = sorted(
    p for p in DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() == '.wav'
)

base_columns = [
    'original_path', 'original_name', 'relative_path',
    'new_name', 'new_path', 'new_relative_path',
    'xx', 'type', 'yy', 'rec', 'zzzz',
    'action', 'error',
]
rows = []

for wav_path in wav_files:
    rel = wav_path.resolve().relative_to(data_root_resolved)
    row = {
        'original_path': str(wav_path),
        'original_name': wav_path.name,
        'relative_path': rel.as_posix(),
        'new_name': None,
        'new_path': None,
        'new_relative_path': None,
        'xx': None,
        'type': None,
        'yy': None,
        'rec': None,
        'zzzz': None,
        'action': 'error',
        'error': None,
    }

    try:
        new_name, parts = _build_new_name(wav_path)
        new_path = wav_path.with_name(new_name)
        new_rel = new_path.resolve().relative_to(data_root_resolved)

        row.update(parts)
        row['new_name'] = new_name
        row['new_path'] = str(new_path)
        row['new_relative_path'] = new_rel.as_posix()
        row['action'] = 'unchanged' if wav_path.name == new_name else 'rename'
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'

    rows.append(row)

PREVIEW_DF = pd.DataFrame(rows, columns=base_columns)

rename_count = int((PREVIEW_DF['action'] == 'rename').sum()) if not PREVIEW_DF.empty else 0
unchanged_count = int((PREVIEW_DF['action'] == 'unchanged').sum()) if not PREVIEW_DF.empty else 0
error_count = int(PREVIEW_DF['error'].notna().sum()) if not PREVIEW_DF.empty else 0

print(f'Total WAV files discovered: {len(PREVIEW_DF)}')
print(f'Rename candidates: {rename_count}')
print(f'Unchanged: {unchanged_count}')
print(f'Errors: {error_count}')

display_cols = ['relative_path', 'original_name', 'new_name', 'new_relative_path', 'action', 'error']
display(PREVIEW_DF[display_cols].head(DISPLAY_LIMIT))


DATA_ROOT: /home/mitch/development/raccoon-ball/datasets/valve
Total WAV files discovered: 12510
Rename candidates: 12510
Unchanged: 0
Errors: 0


,relative_path,original_name,new_name,new_relative_path,action,error
0,+0db-valve/valve/id_00/abnormal/00000000.wav,00000000.wav,+0-valve-00-ab-0000.wav,+0db-valve/valve/id_00/abnormal/+0-valve-00-ab...,rename,None
1,+0db-valve/valve/id_00/abnormal/00000001.wav,00000001.wav,+0-valve-00-ab-0001.wav,+0db-valve/valve/id_00/abnormal/+0-valve-00-ab...,rename,None
2,+0db-valve/valve/id_00/abnormal/00000002.wav,00000002.wav,+0-valve-00-ab-0002.wav,+0db-valve/valve/id_00/abnormal/+0-valve-00-ab...,rename,None
3,+0db-valve/valve/id_00/abnormal/00000003.wav,00000003.wav,+0-valve-00-ab-0003.wav,+0db-valve/valve/id_00/abnormal/+0-valve-00-ab...,rename,None
4,+0db-valve/valve/id_00/abnormal/00000004.wav,00000004.wav,+0-valve-00-ab-0004.wav,+0db-valve/valve/id_00/abnormal/+0-valve-00-ab...,rename,None
...,...,...,...,...,...,...
195,+0db-valve/valve/id_00/normal/00000076.wav,00000076.wav,+0-valve-00-nm-0076.wav,+0db-valve/valve/id_00/normal/+0-valve-00-nm-0...,rename,None
196,+0db-valve/valve/id_00/normal/00000077.wav,00000077.wav,+0-valve-00-nm-0077.wav,+0db-valve/valve/id_00/normal/+0-valve-00-nm-0...,rename,None
197,+0db-valve/valve/id_00/normal/00000078.wav,00000078.wav,+0-valve-00-nm-0078.wav,+0db-valve/valve/id_00/normal/+0-valve-00-nm-0...,rename,None
198,+0db-valve/valve/id_00/normal/00000079.wav,00000079.wav,+0-valve-00-nm-0079.wav,+0db-valve/valve/id_00/normal/+0-valve-00-nm-0...,rename,None


In [9]:
if 'PREVIEW_DF' not in globals():
    raise RuntimeError('Run the preview cell first to build PREVIEW_DF.')

if not PROCESS:
    print('PROCESS is False. No files were renamed.')
else:
    errors_df = PREVIEW_DF[PREVIEW_DF['error'].notna()]
    if not errors_df.empty:
        raise RuntimeError(f'Cannot rename while preview has errors. error_rows={len(errors_df)}')

    candidates = PREVIEW_DF[PREVIEW_DF['action'] == 'rename'].copy()
    if candidates.empty:
        print('No rename candidates found. Nothing to do.')
    else:
        source_paths = [Path(p) for p in candidates['original_path']]
        dest_paths = [Path(p) for p in candidates['new_path']]

        dest_series = pd.Series([str(p) for p in dest_paths])
        duplicate_dest = dest_series[dest_series.duplicated(keep=False)]
        if not duplicate_dest.empty:
            raise RuntimeError(
                'Duplicate destination names detected. Aborting. Example duplicates: '
                + ', '.join(sorted(duplicate_dest.unique())[:10])
            )

        source_set = {p.resolve() for p in source_paths}
        blocking = []
        for dst in dest_paths:
            dst_resolved = dst.resolve()
            if dst_resolved.exists() and dst_resolved not in source_set:
                blocking.append(str(dst_resolved))

        if blocking:
            raise RuntimeError(
                'Destination already exists and is not part of this rename set. Aborting. Example: '
                + ', '.join(blocking[:10])
            )

        staged = []
        completed = []
        try:
            for src, dst in zip(source_paths, dest_paths):
                tmp = src.with_name(f'.__rename_tmp__{uuid4().hex}{src.suffix}')
                src.rename(tmp)
                staged.append((tmp, src, dst))

            for tmp, src, dst in staged:
                tmp.rename(dst)
                completed.append((src, dst))

        except Exception as exc:
            print(f'Rename failed: {type(exc).__name__}: {exc}')
            print('Attempting rollback...')
            for src, dst in reversed(completed):
                if dst.exists() and not src.exists():
                    dst.rename(src)
            for tmp, src, dst in reversed(staged):
                if tmp.exists() and not src.exists():
                    tmp.rename(src)
            raise

        print(f'Rename complete. Files renamed: {len(completed)}')
        result_df = pd.DataFrame({
            'original_path': [str(src) for src, _ in completed],
            'new_path': [str(dst) for _, dst in completed],
            'original_name': [src.name for src, _ in completed],
            'new_name': [dst.name for _, dst in completed],
        })
        display(result_df.head(DISPLAY_LIMIT))


Rename complete. Files renamed: 12510


,original_path,new_path,original_name,new_name
0,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,+0-valve-00-ab-0000.wav
1,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000001.wav,+0-valve-00-ab-0001.wav
2,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000002.wav,+0-valve-00-ab-0002.wav
3,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000003.wav,+0-valve-00-ab-0003.wav
4,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000004.wav,+0-valve-00-ab-0004.wav
...,...,...,...,...
195,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000076.wav,+0-valve-00-nm-0076.wav
196,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000077.wav,+0-valve-00-nm-0077.wav
197,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000078.wav,+0-valve-00-nm-0078.wav
198,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,00000079.wav,+0-valve-00-nm-0079.wav
